# Count synthetic contribution records without counting donors twice

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Learn why transaction counts, unique contributors and net receipts answer different questions. Use Decimal arithmetic, retain negative adjustments, and apply an explicit synthetic amendment policy.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses Python standard library and pandas. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Define transactions with deliberate edge cases

This is a teaching schema inspired by campaign-finance transaction data, not an FEC import format. `transaction_id` identifies a synthetic contribution event; `revision` is our example's ordering field. The FEC bulk schema has its own amendment, filing and transaction fields. Do not apply this simplified key to real filings without consulting those rules.

In [ ]:
from decimal import Decimal, InvalidOperation, localcontext

import pandas as pd

rows = [
    {
        "transaction_id": "t1",
        "revision": 1,
        "contributor_id": "p1",
        "committee": "committee-A",
        "amount": "100.00",
    },
    {
        "transaction_id": "t1",
        "revision": 2,
        "contributor_id": "p1",
        "committee": "committee-A",
        "amount": "125.00",
    },
    {
        "transaction_id": "t2",
        "revision": 1,
        "contributor_id": "p1",
        "committee": "committee-A",
        "amount": "25.00",
    },
    {
        "transaction_id": "t3",
        "revision": 1,
        "contributor_id": "p2",
        "committee": "committee-A",
        "amount": "50.00",
    },
    {
        "transaction_id": "t4",
        "revision": 1,
        "contributor_id": "p2",
        "committee": "committee-A",
        "amount": "-10.00",
    },
    {
        "transaction_id": "t5",
        "revision": 1,
        "contributor_id": "p3",
        "committee": "committee-B",
        "amount": "80.00",
    },
]
pd.DataFrame(rows)

## 2. Resolve amendments deterministically

Conflicting duplicate versions should stop processing instead of silently selecting whichever record arrived last. A repeat of the identical record is safe to ignore. Validate all rows before retaining the highest revision for each transaction.
This teaching schema accepts finite monetary values with cent precision and absolute amount below `10**26`. Use a fixed local Decimal precision for validation, then aggregate Python integer cents and construct exact Decimal totals. A Decimal type alone does not prevent rounding: additions still use the active precision context. The explicit per-record range prevents unexpectedly enormous exponents from entering the integer conversion.


In [ ]:
def latest_transactions(records):
    versions = {}
    for item in records:
        row = dict(item)
        if not all(
            isinstance(row.get(k), str) and row[k]
            for k in ("transaction_id", "contributor_id", "committee")
        ):
            raise ValueError("Identifiers must be nonempty strings")
        if type(row.get("revision")) is not int or row["revision"] < 1:
            raise ValueError("Revision must be a positive integer")
        try:
            amount = Decimal(row["amount"])
        except (InvalidOperation, TypeError, ValueError):
            raise ValueError("Amount must be a valid decimal value") from None
        if not amount.is_finite() or amount.copy_abs() >= Decimal("1e26"):
            raise ValueError("Amount must be finite with absolute value below 10**26")
        with localcontext() as context:
            context.prec = 28  # Up to 26 whole-number digits plus two cents digits.
            rounded = amount.quantize(Decimal("0.01"))
        if amount != rounded:
            raise ValueError("Amount must have at most two decimal places")
        amount = rounded
        row["amount"] = amount
        key = (row["transaction_id"], row["revision"])
        if key in versions and versions[key] != row:
            raise ValueError("Conflicting records for the same revision")
        versions[key] = row
    latest = {}
    for row in versions.values():
        key = row["transaction_id"]
        if key not in latest or row["revision"] > latest[key]["revision"]:
            latest[key] = row
    return sorted(latest.values(), key=lambda row: row["transaction_id"])


clean = latest_transactions(rows)
assert len(clean) == 5
assert latest_transactions(rows + [rows[0]]) == clean
pd.DataFrame(clean)

## 3. Report three distinct measures

A contributor with two positive transactions and a refund is still one contributor. Net receipts include refunds; gross positive receipts do not. These figures are not a count of eligible voters, households, or supporters.

In [ ]:
def exact_money_total(amounts):
    total_cents = 0
    for amount in amounts:
        numerator, denominator = amount.as_integer_ratio()
        cents, remainder = divmod(numerator * 100, denominator)
        if remainder:
            raise ValueError("Aggregate inputs must have exact cent precision")
        total_cents += cents
    # Decimal construction from text is exact and ignores ambient arithmetic precision.
    sign = "-" if total_cents < 0 else ""
    magnitude = abs(total_cents)
    return Decimal(f"{sign}{magnitude // 100}.{magnitude % 100:02d}")


def summarize(records):
    result = []
    for committee in sorted({row["committee"] for row in records}):
        group = [row for row in records if row["committee"] == committee]
        result.append(
            {
                "committee": committee,
                "transaction_count": len(group),
                "unique_contributors": len({row["contributor_id"] for row in group}),
                "positive_receipts": exact_money_total(
                    row["amount"] for row in group if row["amount"] > 0
                ),
                "net_receipts": exact_money_total(row["amount"] for row in group),
            }
        )
    return result


summary = summarize(clean)
assert summary[0]["transaction_count"] == 4
assert summary[0]["unique_contributors"] == 2
assert summary[0]["positive_receipts"] == Decimal("200.00")
assert summary[0]["net_receipts"] == Decimal("190.00")
pd.DataFrame(summary)

## 4. Check failure behavior and order independence

A robust aggregate should not change when a source batch is shuffled. Deliberately contradictory amendments must fail before a misleading total is reported.

In [ ]:
assert latest_transactions(list(reversed(rows))) == clean
assert summarize([]) == []
conflict = dict(rows[0], amount="999.00")
try:
    latest_transactions(rows + [conflict])
except ValueError:
    print("Conflicting amendment rejected.")
else:
    raise AssertionError("Conflict was silently accepted")
assert sum((row["net_receipts"] for row in summary), Decimal("0")) == Decimal("270.00")
# The largest accepted cent values expose rounding in a naive Decimal sum.
large_rows = [
    dict(rows[0], transaction_id=f"large-{index}", amount="99999999999999999999999999.99")
    for index in range(2)
]
with localcontext() as context:
    context.prec = 6  # A caller's context must not change accepted cents or totals.
    large_summary = summarize(latest_transactions(large_rows))
    assert large_summary[0]["net_receipts"] == Decimal("199999999999999999999999999.98")
    assert large_summary[0]["positive_receipts"] == Decimal("199999999999999999999999999.98")
    assert exact_money_total([Decimal("0.01"), Decimal("-0.02")]) == Decimal("-0.01")
for invalid_amount in ("NaN", "Infinity", "0.001", "1e26", "1e999999", "not-a-number"):
    try:
        latest_transactions([dict(rows[0], amount=invalid_amount)])
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid or out-of-range monetary amount accepted")
print("Exact cent totals and six monetary boundary checks passed.")

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

Real contributor identity resolution is difficult: matching names alone can merge different people or split one person. This example uses invented stable IDs and cannot validate actual FEC filing/amendment semantics. No political conclusion follows from these synthetic totals.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [FEC individual contribution file fields and amendment indicators](https://www.fec.gov/campaign-finance-data/contributions-individuals-file-description/)
- [Python Decimal arithmetic](https://docs.python.org/3/library/decimal.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.